In [1]:
from langchain_groq import ChatGroq

In [2]:
llm = ChatGroq(
    temperature=0,

    groq_api_key='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',
    model="llama-3.1-8b-instant",
    max_tokens=None,
    timeout=None,
    max_retries=2


)
response = llm.invoke("who is jay alammar?")   
print(response)

content='Jay Alammar is a designer and writer who is best known for creating the popular "Diagrams" series on Medium, which explores the inner workings of various technologies and concepts through simple, intuitive diagrams.\n\nJay Alammar\'s work focuses on explaining complex ideas in a clear and concise manner, often using visual aids to help illustrate key concepts. His diagrams have been widely shared and praised for their ability to simplify complex topics and make them more accessible to a broad audience.\n\nSome of the topics Jay Alammar has covered in his "Diagrams" series include natural language processing, deep learning, and recommender systems. His work has been widely read and shared within the tech industry, and he is considered one of the leading voices in the field of technical communication and explanation.\n\nJay Alammar is also the founder of the popular newsletter "The Diagrams Newsletter", which delivers a weekly dose of insightful and visually engaging explanation

### change the job career link as itmight get filled over a time

In [4]:
#its webscraping time
# we are using langchain library and the document loaders(WebBaseLoader) for webscraping
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://careers.nike.com/lead-engineer/job/R-55681")
page_data = loader.load().pop().page_content
print(page_data)






















Lead Engineer










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Lan

In [9]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data': page_data})
type(res.content)

str

In [10]:
print(res.content)

```json
[
  {
    "role": "Lead Engineer",
    "experience": "5 years of progressive postbaccalaureate experience in the job offered or in a Engineering-related occupation",
    "skills": [
      "Visual Studio or Visual Code",
      "C# Code Development",
      "net Code Development",
      "Bartender Label Software Integration",
      "Object-oriented design",
      "Service based architecture",
      "Scale Agile Framework Development and Practices",
      "SQL Server Management Studio",
      "Android Scanner Development",
      "Integration to Zebra, Honeywell, and Avery Label Printers",
      "Application Security Design through OAuth"
    ],
    "description": "Develop, codes/configure, test programs/systems and solutions problem in order to meet designed digital product specification and direction. Technical development of digital product(s) that meets the consumer needs as defined by product managers and aligns with architectural standards. Review, analyze, and modify programm

### str to json 

In [11]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Lead Engineer',
  'experience': '5 years of progressive postbaccalaureate experience in the job offered or in a Engineering-related occupation',
  'skills': ['Visual Studio or Visual Code',
   'C# Code Development',
   'net Code Development',
   'Bartender Label Software Integration',
   'Object-oriented design',
   'Service based architecture',
   'Scale Agile Framework Development and Practices',
   'SQL Server Management Studio',
   'Android Scanner Development',
   'Integration to Zebra, Honeywell, and Avery Label Printers',
   'Application Security Design through OAuth'],
  'description': 'Develop, codes/configure, test programs/systems and solutions problem in order to meet designed digital product specification and direction. Technical development of digital product(s) that meets the consumer needs as defined by product managers and aligns with architectural standards. Review, analyze, and modify programming systems, including coding, testing, debugging, and installin

In [12]:

type(json_res)

list

In [15]:
dict_json_res = dict(json_res)
print(dict_json_res)


ValueError: dictionary update sequence element #0 has length 4; 2 is required

In [13]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [18]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [20]:
job = json_res

In [ ]:
jobs_by_role = {item['role']: item for item in json_res }

In [31]:
job = jobs_by_role['Lead Engineer']
print(job['skills'])

['Visual Studio or Visual Code', 'C# Code Development', 'net Code Development', 'Bartender Label Software Integration', 'Object-oriented design', 'Service based architecture', 'Scale Agile Framework Development and Practices', 'SQL Server Management Studio', 'Android Scanner Development', 'Integration to Zebra, Honeywell, and Avery Label Printers', 'Application Security Design through OAuth']


In [32]:
links = collection.query(
    query_texts=job['skills'],
    n_results=2
).get('metadatas', [])
links

[[{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}

In [33]:
links = collection.query(
    query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/angular-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}

In [34]:
from langchain import PromptTemplate

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are Slanzer, a Business Development Executive at slnX. 
    slnX is an AI & Software Consulting firm committed to helping organizations accelerate growth, streamline operations, and 
    reduce costs through bespoke automation and AI‑driven solutions.
    Over the years, we’ve partnered with industry leaders to deliver measurable ROI, 
    scalable architectures, and rapid time‑to‑value.

    Write a concise, highly personalized cold email (no preamble) to the recipient about the opportunity described in the job description. 
    Highlight slnX’s core capabilities that directly address their needs, 
    reference relevant portfolio items from the following links to build credibility, and include a clear next step (e.g., suggest a brief call).

    ### PORTFOLIO LINKS:
    {link_list}

    ### EMAIL (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Unlock Your Team's Potential with AI-Driven Solutions

Dear [Recipient's Name],

I came across your job description for a Lead Engineer at [Company Name], and I was impressed by the scope of responsibilities and the technical expertise required for the role. As a Business Development Executive at slnX, an AI & Software Consulting firm, I believe our core capabilities align perfectly with your needs.

Our team has extensive experience in developing scalable architectures, integrating with various label printers (Zebra, Honeywell, and Avery), and designing secure applications using OAuth. We've successfully delivered projects that meet the consumer needs of our clients, and I'd love to share some examples from our portfolio:

- Our Angular portfolio showcases our expertise in building scalable and maintainable applications: https://example.com/angular-portfolio
- Our ML-Python portfolio highlights our ability to integrate AI-driven solutions: https://example.com/ml-python-portfo